In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2, RFE, SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

In [2]:
real = pd.read_csv("True.csv")
fake = pd.read_csv("Fake.csv")


In [3]:
real["label"] = "Real"
fake["label"] = "Fake"

data = pd.concat([real, fake], ignore_index=True)


In [4]:
X = data["text"]
y = data["label"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [7]:
# apply selectKBest
select_kbest = SelectKBest(score_func=chi2, k=3000)

X_train_kbest = select_kbest.fit_transform(X_train_tfidf, y_train)
X_test_kbest = select_kbest.transform(X_test_tfidf)

model1 = LinearSVC()
model1.fit(X_train_kbest, y_train)

y_pred1 = model1.predict(X_test_kbest)
accuracy1 = accuracy_score(y_test, y_pred1)

print("SelectKBest Accuracy:", accuracy1)

SelectKBest Accuracy: 0.9933184855233853


In [8]:
# Apply Recursive Feature Elimination (RFE)
rfe = RFE(estimator=LinearSVC(), n_features_to_select=3000)

X_train_rfe = rfe.fit_transform(X_train_tfidf, y_train)
X_test_rfe = rfe.transform(X_test_tfidf)

model2 = LinearSVC()
model2.fit(X_train_rfe, y_train)

y_pred2 = model2.predict(X_test_rfe)
accuracy2 = accuracy_score(y_test, y_pred2)

print("RFE Accuracy:", accuracy2)

RFE Accuracy: 0.9947661469933184


In [9]:
# Apply SelectFromModel
selector = SelectFromModel(LinearSVC(), max_features=3000)

X_train_selected = selector.fit_transform(X_train_tfidf, y_train)
X_test_selected = selector.transform(X_test_tfidf)

model3 = LinearSVC()
model3.fit(X_train_selected, y_train)

y_pred3 = model3.predict(X_test_selected)
accuracy3 = accuracy_score(y_test, y_pred3)

print("SelectFromModel Accuracy:", accuracy3)

SelectFromModel Accuracy: 0.9934298440979955


In [10]:
# compare all techniques
comparison = pd.DataFrame({
    "Feature Selection Technique": [
        "SelectKBest",
        "RFE",
        "SelectFromModel"
    ],
    "Accuracy": [
        accuracy1,
        accuracy2,
        accuracy3
    ]
})

comparison

,Feature Selection Technique,Accuracy
0,SelectKBest,0.993318
1,RFE,0.994766
2,SelectFromModel,0.993430


In [11]:
best_technique = comparison.loc[comparison["Accuracy"].idxmax()]

print("Best Feature Selection Technique:")
print(best_technique)

Best Feature Selection Technique:
Feature Selection Technique         RFE
Accuracy                       0.994766
Name: 1, dtype: object
